# Thử scoring Controller v4 trên Tesla T4

Chỉnh một cell CONFIG rồi chọn **Runtime → Run all**. Notebook clone repo private bằng Secret `GITHUB_TOKEN`, khóa commit, build/test, đo Reference và đủ P0–P5. Model, feature, post-gate và QoR giữ nguyên. Đây là đối chứng trên development đã xem; không dùng để tuyên bố đánh giá độc lập.

Cần checkpoint v4 photo và ảnh P6 PPM 512×512 trên Drive. Manifest phải trỏ tới ảnh thật trong runtime này. Xem `docs/controller-score-colab-run-guide.md`. Bản source chứa notebook phải được Đế push lên GitHub trước khi clone; agent không tự push.


In [ ]:
# CONFIG — the only cell to edit. Never put a token here.
CONFIG = {
    "REPOSITORY": "OWNER/REPO",
    "GIT_REF": "de/optimize-controller",  # Branch or full commit SHA already on GitHub.
    "MOUNT_DRIVE": True,
    "MODE": "FULL",  # FULL runs an automatic smoke first; SMOKE exports 3-pair diagnostics only.
    "POLICY": "/content/drive/MyDrive/controller-v4/final-checkpoint/v4-fast-ablations/photo/policy.json",
    "SPLIT": "/content/drive/MyDrive/controller-v4/final-checkpoint/v4-inputs/split.json",
    "CALIBRATION_MANIFEST": "/content/drive/MyDrive/controller-v4/calibration-relocated.json",
    "EVALUATION_MANIFEST": "/content/drive/MyDrive/controller-v4/validation-relocated.json",
    "OUTPUT_ROOT": "/content/drive/MyDrive/controller-score-ablation",
    "ORACLE_SWEEP_DIR": "",  # Optional: complete compatible runs/qor/configs/receipt, never an automatic sweep.
}


## 1. Kiểm GPU và lưu dữ liệu

Runtime phải có đúng một Tesla T4 và nvcc. Mount Drive để checkpoint còn sau khi mất runtime. Nếu dùng ổ `/content`, cần tải ZIP trước khi phiên kết thúc. Mọi lượt đo dùng seed `20261101`, hai warmup và mười repeat.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import uuid

if CONFIG["MODE"] not in ("SMOKE", "FULL"):
    raise RuntimeError("CONFIG.MODE phải là SMOKE hoặc FULL")
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,uuid", "--format=csv,noheader"], text=True).strip().splitlines()
    if len(gpu) != 1 or "Tesla T4" not in gpu[0]:
        raise RuntimeError("Cần đúng một GPU Tesla T4; chọn lại runtime T4 rồi Run all.")
    subprocess.run(["nvcc", "--version"], check=True)
except FileNotFoundError:
    raise RuntimeError("Runtime chưa có GPU/CUDA. Chọn GPU Tesla T4 rồi Run all.") from None
print("GPU:", gpu[0], "SEED: 20261101; WARMUP: 2; REPEATS: 10")
if CONFIG["MOUNT_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
# Ephemeral /content identity survives reruns in this runtime, not a replacement VM.
_session_path = Path("/content/controller-score-runtime-id")
if not _session_path.exists():
    _session_path.write_text(uuid.uuid4().hex)
CONFIG["SESSION_ID"] = _session_path.read_text().strip()


## 2. Clone private repo và khóa commit

Trong Colab, mở biểu tượng chìa khóa **Secrets**, tạo `GITHUB_TOKEN` và bật **Notebook access**. Token Fine-grained chỉ cần quyền **Contents: Read-only** cho đúng repo. Cell không in token, không ghi token vào URL hoặc Git config. Token chỉ được truyền tới tiến trình Git qua bộ nhớ môi trường con và ASKPASS tạm; file ASKPASS chỉ chứa code đọc biến môi trường.


In [ ]:
import re
import shutil
import tempfile


def checkout_private(repository, ref, workspace, get_secret):
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository) or repository == "OWNER/REPO":
        raise RuntimeError("Điền CONFIG.REPOSITORY theo dạng OWNER/REPO.")
    if not ref or ref.startswith("-") or any(c in ref for c in "\r\n"):
        raise RuntimeError("Điền CONFIG.GIT_REF bằng branch hoặc commit hợp lệ đã push.")
    checkout = Path(workspace) / "controller-score-source"
    lock = checkout / ".git/score-checkout.json"
    if lock.exists():
        stored = json.loads(lock.read_text())
        actual = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
        if stored != {"repository": repository, "ref": ref, "revision": actual}:
            raise RuntimeError("Checkout đã khóa khác CONFIG. Dùng runtime mới để đổi repo/ref; không trộn source giữa checkpoint.")
        return checkout, actual
    if checkout.exists():
        raise RuntimeError("Checkout dở dang. Xóa thư mục /content/controller-score-source trong Files rồi Run all; giữ nguyên checkpoint trên Drive.")
    try:
        token = get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Không đọc được GITHUB_TOKEN. Mở Secrets, thêm token và bật Notebook access; kiểm quyền repository.") from None
    if not isinstance(token, str) or not token.strip():
        token = None
        raise RuntimeError("GITHUB_TOKEN trống. Tạo Fine-grained PAT Contents: Read-only cho đúng repo trong Colab Secrets.")
    child_env = {k:v for k,v in os.environ.items() if not k.startswith("GIT_") and k not in ("GITHUB_TOKEN", "GH_TOKEN")}
    try:
        with tempfile.TemporaryDirectory(prefix="score-git-auth-") as auth:
            askpass = Path(auth) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['SCORE_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            child_env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0", SCORE_GITHUB_TOKEN=token)
            args = ["git", "-c", "credential.helper=", "-c", "core.askPass="+str(askpass), "-c", "http.extraHeader="]
            commands = [args+["init", str(checkout)],
                args+["-C", str(checkout), "remote", "add", "origin", "https://github.com/"+repository+".git"],
                args+["-C", str(checkout), "fetch", "--depth=1", "origin", ref],
                args+["-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"]]
            for command in commands:
                completed = subprocess.run(command, env=child_env, capture_output=True, text=True)
                if completed.returncode:
                    # Suppress all network output; no credential-bearing output reaches a cell/log.
                    raise RuntimeError("Clone/checkout thất bại. Kiểm OWNER/REPO, ref đã push, Contents: Read-only và approval của tổ chức; không đổi sang token ghi hoặc URL chứa token.")
    finally:
        token = None
        child_env.pop("SCORE_GITHUB_TOKEN", None)
        child_env.pop("GIT_ASKPASS", None)
    revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Không khóa được full commit SHA")
    lock.write_text(json.dumps({"repository": repository, "ref": ref, "revision": revision}))
    return checkout, revision


from google.colab import userdata
SOURCE, REVISION = checkout_private(CONFIG["REPOSITORY"], CONFIG["GIT_REF"], "/content", userdata.get)
os.chdir(SOURCE)
if not (SOURCE/"tools/controller/score_ablation.py").is_file():
    raise RuntimeError("Ref chưa chứa implementation score ablation. Push commit đã bàn giao rồi sửa GIT_REF.")
print("LOCKED COMMIT:", REVISION)


## 3. Dependencies và kiểm input trước build

Cài requirements hiện có của repo, gồm XGBoost để đọc model và chạy unit tests. Không gọi trainer cho model thí nghiệm. Nếu checkpoint hoặc ảnh thiếu, cell dừng và chỉ rõ đường dẫn; không thay dataset/model. Các manifest được yêu cầu thuộc calibration/validation của split đã khóa; kết quả được ghi nhãn diagnostic trên development đã xem.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake", "nbformat",
    "-r", "tools/controller/requirements-learn.txt", "-r", "tools/controller/requirements-report.txt"], check=True)
sys.path.insert(0, str(SOURCE))
from tools.controller.score_ablation import preflight
preflight(CONFIG)
print("INPUT/SCHEMA/PROVENANCE PASS")


## 4. Build, tests, smoke, sáu policy và báo cáo

Run all tự chạy toàn bộ: Release `sm_75`, CTest, pytest, profile Reference trên calibration, khóa P0–P5, smoke/parity, rồi FULL. P0–P2 giữ score cũ, P3–P5 dùng reuse approximation; mỗi nhóm dùng pre-threshold 0.85/0.70/0.50. Gate giữ 0.65. Thứ tự mode được xoay theo batch; mỗi mode/batch có receipt/hash và ZIP checkpoint trên Drive. Khác runtime/GPU không được nối thành paired campaign.

Profile giữ nguyên S5, S6 và host overhead; R-only trừ S1–S4, đổi K trừ S1, đổi S không giảm. Đây là score xấp xỉ: `1-p_pre` là failure proxy; fixed latency của candidate đã gồm S6 nên score vẫn có sai lệch ở branch reject. Kết luận dựa trên E2E đo thật. Oracle chỉ có khi sweep đủ provenance tương thích; thiếu thì báo UNAVAILABLE và không tự chạy sweep 48 cấu hình.


In [ ]:
from tools.controller.score_ablation import run
RESULT_ZIP = run(CONFIG)


## 5. Tải ZIP và gửi lại

Bảng `comparison.csv`, `summary.json`, raw CSV, attempt traces, diagnostics, policy và hash nằm trong ZIP. Gửi ZIP cùng **LOCKED COMMIT** trong chat tiếp theo. Không tự chọn policy thắng hoặc đổi default từ bảng này. Nếu cell trước lỗi, ZIP partial vẫn nằm dưới OUTPUT_ROOT; mở Files hoặc Drive để tải log/checkpoint.


In [ ]:
from google.colab import files
print("Gửi ZIP:", RESULT_ZIP)
print("Git commit:", REVISION)
files.download(str(RESULT_ZIP))
